<a href="https://colab.research.google.com/github/AMJAMAITHILI/DL_LAB/blob/main/WEEK5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement the MLP using the Types of Regularization Techniques.


**L2 Regularization**


In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models, regularizers

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the input data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

#FIX: Reshape 28x28 images into 784-length vectors
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Define a simple model with L2 regularization
def build_l2_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu',
                     input_shape=(784,),
                     kernel_regularizer=regularizers.l2(0.01)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(0.01)),
        layers.Dense(10, activation='softmax')
    ])
    return model

model_l2 = build_l2_model()

model_l2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model_l2.fit(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=32
)

# Evaluate the model
test_loss, test_acc = model_l2.evaluate(x_test, y_test)

print(f"Test accuracy: {test_acc:.4f}")


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8811 - loss: 0.8527 - val_accuracy: 0.9224 - val_loss: 0.5564
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9155 - loss: 0.5414 - val_accuracy: 0.9325 - val_loss: 0.4708
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9254 - loss: 0.4787 - val_accuracy: 0.9362 - val_loss: 0.4396
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9325 - loss: 0.4387 - val_accuracy: 0.9494 - val_loss: 0.3878
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9375 - loss: 0.4112 - val_accuracy: 0.9518 - val_loss: 0.3702
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9425 - loss: 0.3851 - val_accuracy: 0.9472 - val_loss: 0.3651
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9452 - loss: 0.3688 - val_accuracy: 0.9523 - val_loss: 0.3466
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9466 - loss: 0.3538 -

**Dataset Augmentation**



In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

#Reshape to (samples, 28, 28, 1) for augmentation
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

#Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False,
    vertical_flip=False
)

datagen.fit(x_train)

# Build Model (with L2 regularization)
def build_l2_model():
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(0.01)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(0.01)),
        layers.Dense(10, activation='softmax')
    ])
    return model

model_l2 = build_l2_model()

model_l2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

#Train using augmented data
history = model_l2.fit(
    datagen.flow(x_train, y_train, batch_size=32),
    epochs=10,
    validation_data=(x_test, y_test),
    steps_per_epoch=len(x_train) // 32,verbose=0
)

#Evaluate model
test_loss, test_acc = model_l2.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9421 - loss: 0.4371
Test accuracy: 0.9421


**Parameter sharing and tying**


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Add channel dimension
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)

print("Training shape:", x_train.shape)

# CNN model
model = models.Sequential([
    layers.Input(shape=(28,28,1)),
    layers.Conv2D(filters=1, kernel_size=(3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model structure
model.summary()

# Train
model.fit(x_train, y_train, epochs=3, batch_size=64)

# Test accuracy
loss, accuracy = model.evaluate(x_test, y_test)
print("Test accuracy:", accuracy)

# Show filter parameters (demonstrates parameter sharing)
weights = model.layers[0].get_weights()[0]

print("Filter shape:", weights.shape)
print("Total filter parameters:", weights.size)

print("\nExplanation:")
print("The 3x3 filter has only 9 weights.")
print("This same filter slides across the entire image.")
print("Hence the same parameters are reused everywhere (parameter sharing).")

Training shape: (60000, 28, 28, 1)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 26, 26, 1)      │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 676)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 10)             │         6,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,780 (26.48 KB)

 Trainable params: 6,780 (26.48 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7941 - loss: 0.7419
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9115 - loss: 0.3090
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.9195 - loss: 0.2840
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9206 - loss: 0.2716
Test accuracy: 0.9205999970436096
Filter shape: (3, 3, 1, 1)
Total filter parameters: 9

Explanation:
The 3x3 filter has only 9 weights.
This same filter slides across the entire image.
Hence the same parameters are reused everywhere (parameter sharing).


**Adding noise to the inputs and outputs**


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape to 784 vector
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# ===============================
# Add Noise to INPUTS
# ===============================
noise_factor = 0.2
x_train_noisy = x_train + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=x_train.shape)

# Keep pixel values between 0 and 1
x_train_noisy = np.clip(x_train_noisy, 0., 1.)

# ===============================
# 🔹 Add Noise to OUTPUTS (Label Noise)
# Randomly flip 10% of labels
# ===============================
noise_ratio = 0.1
num_samples = int(noise_ratio * y_train.shape[0])

random_indices = np.random.choice(y_train.shape[0], num_samples, replace=False)

for idx in random_indices:
    random_label = np.random.randint(0, 10)
    y_train[idx] = to_categorical(random_label, 10)

# ===============================
# Build Model (same as original)
# ===============================
def build_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_model()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train model using noisy inputs and noisy labels
history = model.fit(
    x_train_noisy,
    y_train,
    epochs=20,
    batch_size=256,
    validation_split=0.2
)

# Evaluate on clean test data
test_loss, test_acc = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_acc:.4f}")


Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7398 - loss: 1.0628 - val_accuracy: 0.8272 - val_loss: 0.8111
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8282 - loss: 0.7854 - val_accuracy: 0.8390 - val_loss: 0.7564
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8457 - loss: 0.7235 - val_accuracy: 0.8492 - val_loss: 0.7232
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8575 - loss: 0.6809 - val_accuracy: 0.8546 - val_loss: 0.7110
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8638 - loss: 0.6498 - val_accuracy: 0.8569 - val_loss: 0.6955
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8695 - loss: 0.6242 - val_accuracy: 0.8608 - val_loss: 0.6939
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8735 - loss: 0.6033 - val_accuracy: 0.8598 - val_loss: 0.6857
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8777 - loss: 0.5825 - val_accuracy: 0.

**Early stopping**

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape to 784 vector
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build simple model (NO L2 regularization)
def build_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_model()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 🔹 Early Stopping
earlystop = EarlyStopping(
    monitor='val_accuracy',   # Monitor validation accuracy
    patience=3,               # Stop if no improvement after 3 epochs
    restore_best_weights=True # Restore best model weights
)

epochs = 20  # change this value and check
batch_size = 256   # change this value and check

# Train model with Early Stopping
history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[earlystop]
)

# Evaluate model
test_loss, test_acc = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_acc:.4f}")


Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8465 - loss: 0.5548 - val_accuracy: 0.9273 - val_loss: 0.2611
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9356 - loss: 0.2238 - val_accuracy: 0.9503 - val_loss: 0.1848
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9513 - loss: 0.1686 - val_accuracy: 0.9553 - val_loss: 0.1577
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9598 - loss: 0.1403 - val_accuracy: 0.9578 - val_loss: 0.1459
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9651 - loss: 0.1185 - val_accuracy: 0.9617 - val_loss: 0.1313
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9703 - loss: 0.1019 - val_accuracy: 0.9645 - val_loss: 0.1211
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9739 - loss: 0.0887 - val_accuracy: 0.9656 - val_loss: 0.1125
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9770 - loss: 0.0786 - val_accuracy: 0.

**Ensemble methods**


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Flatten for simple NN
x_train = x_train.reshape(-1,784)
x_test = x_test.reshape(-1,784)

# Function to create model
def create_model():
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


models_list = []

# Train 3 models
for i in range(3):
    print("\nTraining model", i+1)

    m = create_model()
    m.fit(x_train, y_train, epochs=3, batch_size=64, verbose=0)

    loss, acc = m.evaluate(x_test, y_test, verbose=0)
    print("Model", i+1, "accuracy:", acc)

    models_list.append(m)


# Ensemble prediction
predictions = []

for m in models_list:
    predictions.append(m.predict(x_test))

avg_prediction = np.mean(predictions, axis=0)

ensemble_pred = np.argmax(avg_prediction, axis=1)

ensemble_accuracy = np.mean(ensemble_pred == y_test)

print("\nEnsemble accuracy:", ensemble_accuracy)


Training model 1
Model 1 accuracy: 0.9724000096321106

Training model 2
Model 2 accuracy: 0.9728000164031982

Training model 3
Model 3 accuracy: 0.9702000021934509
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

Ensemble accuracy: 0.9774


**Dropouts**

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape to 784 vector
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build model (Same as before + Dropout added)
def build_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(784,)),
        layers.Dropout(0.3),              # 🔹 Dropout added
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),              # 🔹 Dropout added
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_model()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

epochs = 20
batch_size = 256

# Train model (NO Early Stopping)
history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

# Evaluate model
test_loss, test_acc = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_acc:.4f}")


Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7230 - loss: 0.8665 - val_accuracy: 0.9173 - val_loss: 0.2876
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8832 - loss: 0.3955 - val_accuracy: 0.9388 - val_loss: 0.2095
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9069 - loss: 0.3197 - val_accuracy: 0.9482 - val_loss: 0.1798
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9187 - loss: 0.2781 - val_accuracy: 0.9531 - val_loss: 0.1639
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9282 - loss: 0.2472 - val_accuracy: 0.9576 - val_loss: 0.1461
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9337 - loss: 0.2230 - val_accuracy: 0.9592 - val_loss: 0.1413
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9378 - loss: 0.2112 - val_accuracy: 0.9603 - val_loss: 0.1335
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9422 - loss: 0.1941 - val_accuracy: 0.

 **Explore on your chosen dataset and write your own observation of the best technique and reason**




In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# =====================================================
# Load CIFAR10 Dataset
# =====================================================
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

y_train = tf.keras.utils.to_categorical(y_train,10)
y_test = tf.keras.utils.to_categorical(y_test,10)

# =====================================================
# Base CNN Model (Parameter Sharing happens in Conv2D)
# =====================================================
def base_model(l2=False, dropout=False):

    reg = regularizers.l2(0.001) if l2 else None

    model = models.Sequential([
        layers.Conv2D(32,(3,3),activation='relu',
                      kernel_regularizer=reg,
                      input_shape=(32,32,3)),
        layers.MaxPooling2D(),

        layers.Conv2D(64,(3,3),activation='relu',
                      kernel_regularizer=reg),
        layers.MaxPooling2D(),

        layers.Flatten(),

        layers.Dense(128,activation='relu',
                     kernel_regularizer=reg),

        layers.Dropout(0.5) if dropout else layers.Activation('linear'),

        layers.Dense(10,activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


results = {}

# =====================================================
# 1. L2 REGULARIZATION
# =====================================================
model = base_model(l2=True)

model.fit(x_train,y_train,epochs=5,batch_size=64,verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["L2 Regularization"] = acc


# =====================================================
# 2. DATA AUGMENTATION
# =====================================================
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

model = base_model()

model.fit(datagen.flow(x_train,y_train,batch_size=64),
          epochs=5,verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["Dataset Augmentation"] = acc


# =====================================================
# 3. PARAMETER SHARING
# (Automatically happens in CNN filters)
# =====================================================
model = base_model()

model.fit(x_train,y_train,epochs=5,batch_size=64,verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["Parameter Sharing (CNN)"] = acc


# =====================================================
# 4. ADDING NOISE TO INPUTS
# =====================================================
noise = 0.1 * np.random.normal(size=x_train.shape)
x_train_noisy = np.clip(x_train + noise,0,1)

model = base_model()

model.fit(x_train_noisy,y_train,epochs=5,batch_size=64,verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["Input Noise"] = acc


# =====================================================
# 5. EARLY STOPPING
# =====================================================
model = base_model()

early = EarlyStopping(patience=2,restore_best_weights=True)

model.fit(x_train,y_train,
          epochs=20,
          batch_size=64,
          validation_split=0.2,
          callbacks=[early],
          verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["Early Stopping"] = acc


# =====================================================
# 6. DROPOUT
# =====================================================
model = base_model(dropout=True)

model.fit(x_train,y_train,epochs=5,batch_size=64,verbose=0)

_,acc = model.evaluate(x_test,y_test,verbose=0)
results["Dropout"] = acc


# =====================================================
# 7. ENSEMBLE METHODS
# =====================================================
models_list = []

for i in range(3):
    m = base_model()
    m.fit(x_train,y_train,epochs=3,batch_size=64,verbose=0)
    models_list.append(m)

preds = [m.predict(x_test) for m in models_list]

avg = np.mean(preds,axis=0)

ensemble_pred = np.argmax(avg,axis=1)
true = np.argmax(y_test,axis=1)

ensemble_acc = np.mean(ensemble_pred==true)

results["Ensemble"] = ensemble_acc


# =====================================================
# RESULT TABLE
# =====================================================
table = pd.DataFrame({
    "Technique": list(results.keys()),
    "Test Accuracy": list(results.values())
})

print("\nRegularization Comparison Table\n")
print(table)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step

Regularization Comparison Table

                 Technique  Test Accuracy
0        L2 Regularization         0.6526
1     Dataset Augmentation         0.6553
2  Parameter Sharing (CNN)         0.6786
3              Input Noise         0.6251
4           Early Stopping         0.6978
5                  Dropout         0.6764
6                 Ensemble         0.6817
